# Interim: ablations

Three studies on top of the CREMA-D + RAVDESS interim pipeline:

1. **Backbone ablation (visual).** F1-concat with `enet_b0_8_va_mtl` (1280-D) vs. `mbf_va_mtl` (512-D). Same data, same fusion, same hparams -- the only variable is the visual trunk.
2. **Cross-dataset transfer, RH5.** Train the full suite on CREMA-D, evaluate as-is on RAVDESS. Measures how much the Russell VA mapping + actor-disjoint training generalize across two independent studio-acted corpora.
3. **RAVDESS-trained mirror.** Train the two anchors (visual_only, F1-concat) on RAVDESS directly so we can compare with-adaptation vs. zero-shot in the same table.

Outputs: `results/interim/ablation_backbone.md`, `ablation_cross_dataset.md`, `ablation_ravdess_trained.md`, and a combined `ablations_summary.md`.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("cwd:", Path.cwd())

from omegaconf import OmegaConf
from src.train_fusion import train_from_config
from src.eval_fusion import evaluate_variant

CONFIGS = ROOT / "configs" / "interim"
RESULTS = ROOT / "results" / "interim"
RESULTS.mkdir(parents=True, exist_ok=True)


def train_or_skip(cfg) -> Path:
    """Train if no best.pt exists at cfg.output.results_dir, else return its path."""
    ck = Path(cfg.output.results_dir) / "best.pt"
    if ck.exists():
        print(f"[skip] {ck} exists, reusing")
        return ck
    return train_from_config(cfg)

cwd: C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code


## Ablation 1: visual backbone (enet vs. mbf) under F1-concat

Uses the existing CREMA-D F1 checkpoint (enet, 1280-D) and trains the mbf variant (512-D) fresh. Same val set, same fusion hparams.

In [2]:
# (a) Train F1 with mbf_va_mtl visual trunk (skips if best.pt already exists).
mbf_cfg = OmegaConf.load(CONFIGS / "crema_f1_concat_mbf.yaml")
ck_mbf = train_or_skip(mbf_cfg)

# (b) Re-evaluate the existing enet checkpoint for an apples-to-apples number.
ck_enet = ROOT / "results" / "interim" / "crema_f1_concat" / "best.pt"

m_enet = evaluate_variant(
    checkpoint=ck_enet,
    config_path=CONFIGS / "crema_f1_concat.yaml",
    output_dir=RESULTS / "crema_f1_concat",
)
m_mbf = evaluate_variant(
    checkpoint=ck_mbf,
    config_path=CONFIGS / "crema_f1_concat_mbf.yaml",
    output_dir=RESULTS / "crema_f1_concat_mbf",
)
print("enet:", m_enet["P_MTL@0.5"], "params=", m_enet["trainable_params"])
print("mbf :", m_mbf["P_MTL@0.5"], "params=", m_mbf["trainable_params"])

[fusion=f1_concat] trainable params = 699,242
[f1_concat] ep   1  train=1.4671  val=1.6085  best=1.6085  *
[f1_concat] ep   2  train=0.9954  val=1.7849  best=1.6085
[f1_concat] ep   3  train=0.8212  val=1.8839  best=1.6085
[f1_concat] ep   4  train=0.7141  val=1.8939  best=1.6085
[f1_concat] ep   5  train=0.6352  val=1.9213  best=1.6085
[f1_concat] ep   6  train=0.5728  val=2.0012  best=1.6085
[f1_concat] ep   7  train=0.5224  val=2.0586  best=1.6085
[f1_concat] ep   8  train=0.4804  val=2.2102  best=1.6085
[f1_concat] ep   9  train=0.4433  val=2.3153  best=1.6085
[f1_concat] ep  10  train=0.4079  val=2.2325  best=1.6085
[f1_concat] ep  11  train=0.3818  val=2.3999  best=1.6085
[f1_concat] ep  12  train=0.3562  val=2.4096  best=1.6085
[f1_concat] ep  13  train=0.3414  val=2.5251  best=1.6085
[f1_concat] ep  14  train=0.3211  val=2.6145  best=1.6085
[f1_concat] ep  15  train=0.2983  val=2.7665  best=1.6085
[fusion=f1_concat] checkpoint -> results\interim\crema_f1_concat_mbf\best.pt


c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


[stage3] wrote C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\interim\crema_f1_concat\metrics.md
               ccc_V = 0.7401
               ccc_A = 0.4525
              CCC_VA = 0.5963
       F1_EXPR_macro = 0.4389
            ACC_EXPR = 0.5875
           F1_AU@0.5 = 0.0000
          F1_AU_best = 0.0000
           t_AU_best = 0.5000
           P_MTL@0.5 = 1.0352
          P_MTL_best = 1.0352
             variant = f1_concat
    trainable_params = 995690.0000
[stage3] wrote C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\interim\crema_f1_concat_mbf\metrics.md
               ccc_V = 0.7177
               ccc_A = 0.4145
              CCC_VA = 0.5661
       F1_EXPR_macro = 0.4205
            ACC_EXPR = 0.5641
           F1_AU@0.5 = 0.0000
          F1_AU_best = 0.0000
           t_AU_best = 0.5000
           P_MTL@0.5 = 0.9867
          P_MTL_best = 0.9867
             variant = f1_concat
    trainable_params = 699242.0000
enet: 1.0352028168912704 params= 99569

c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [3]:
import pandas as pd

COLS = ["variant", "ccc_V", "ccc_A", "CCC_VA", "F1_EXPR_macro", "F1_AU@0.5", "P_MTL@0.5", "trainable_params"]
df_bb = pd.DataFrame([
    {"backbone": "enet_b0_8_va_mtl (1280)", **{k: m_enet.get(k, '') for k in COLS}},
    {"backbone": "mbf_va_mtl (512)",        **{k: m_mbf.get(k, '') for k in COLS}},
])
(RESULTS / "ablation_backbone.md").write_text(
    "# Visual backbone ablation (CREMA-D, F1-concat)\n\n" +
    df_bb.to_markdown(index=False, floatfmt=".4f") + "\n",
    encoding="utf-8",
)
df_bb

,backbone,variant,ccc_V,ccc_A,CCC_VA,F1_EXPR_macro,F1_AU@0.5,P_MTL@0.5,trainable_params
0,enet_b0_8_va_mtl (1280),f1_concat,0.740063,0.452515,0.596289,0.438914,0.0,1.035203,995690
1,mbf_va_mtl (512),f1_concat,0.717738,0.414533,0.566135,0.420516,0.0,0.986651,699242


## Ablation 2: CREMA-D -> RAVDESS zero-shot (RH5)

Reuse the trained CREMA-D checkpoint but swap the validation annotation path to RAVDESS. `evaluate_variant` honors whatever `cfg.data.val_annotations` points at, so we override via OmegaConf and point `features_cache` / `aligned_dir` at the RAVDESS caches.

In [4]:
import tempfile

def cross_dataset_cfg(base_yaml: Path, out_yaml: Path) -> Path:
    cfg = OmegaConf.load(base_yaml)
    cfg.data.val_annotations = "data/ravdess/annotations/val.txt"
    cfg.visual.features_cache = "cache/features/ravdess/enet_b0_8_va_mtl"
    cfg.audio.aligned_dir = "cache/features/ravdess/hubert_large_aligned"
    cfg.output.results_dir = str((RESULTS / f"{out_yaml.stem}").resolve())
    OmegaConf.save(cfg, out_yaml)
    return out_yaml

CROSS_DIR = RESULTS / "_cross_cfgs"
CROSS_DIR.mkdir(parents=True, exist_ok=True)

cross_results = {}
for name in [
    "crema_visual_only", "crema_audio_only",
    "crema_f1_concat", "crema_f2_blend", "crema_f3_gate",
    "crema_f4_xattn", "crema_f5_lmf",
]:
    ck_path = RESULTS / name / "best.pt"
    if not ck_path.exists():
        print(f"[skip] {name}: no checkpoint at {ck_path}")
        continue
    new_yaml = cross_dataset_cfg(
        CONFIGS / f"{name}.yaml",
        CROSS_DIR / f"{name}_on_ravdess.yaml",
    )
    m = evaluate_variant(
        checkpoint=ck_path,
        config_path=new_yaml,
        output_dir=RESULTS / f"{name}_on_ravdess",
    )
    cross_results[name] = m

c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


[stage3] wrote C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\interim\crema_visual_only_on_ravdess\metrics.md
               ccc_V = 0.7372
               ccc_A = 0.3984
              CCC_VA = 0.5678
       F1_EXPR_macro = 0.3907
            ACC_EXPR = 0.5067
           F1_AU@0.5 = 0.0000
          F1_AU_best = 0.0000
           t_AU_best = 0.5000
           P_MTL@0.5 = 0.9585
          P_MTL_best = 0.9585
             variant = visual_only
    trainable_params = 179706.0000


c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


[stage3] wrote C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\interim\crema_audio_only_on_ravdess\metrics.md
               ccc_V = 0.0561
               ccc_A = 0.0859
              CCC_VA = 0.0710
       F1_EXPR_macro = 0.1631
            ACC_EXPR = 0.2195
           F1_AU@0.5 = 0.0000
          F1_AU_best = 0.0000
           t_AU_best = 0.5000
           P_MTL@0.5 = 0.2341
          P_MTL_best = 0.2341
             variant = audio_only
    trainable_params = 142998.0000


c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


...

c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


[stage3] wrote C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\interim\crema_f3_gate_on_ravdess\metrics.md
               ccc_V = 0.6761
               ccc_A = 0.3808
              CCC_VA = 0.5284
       F1_EXPR_macro = 0.3649
            ACC_EXPR = 0.4699
           F1_AU@0.5 = 0.0000
          F1_AU_best = 0.0000
           t_AU_best = 0.5000
           P_MTL@0.5 = 0.8933
          P_MTL_best = 0.8933
             variant = f3_gate
    trainable_params = 624168.0000


c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


[stage3] wrote C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\interim\crema_f4_xattn_on_ravdess\metrics.md
               ccc_V = 0.6958
               ccc_A = 0.2743
              CCC_VA = 0.4850
       F1_EXPR_macro = 0.3714
            ACC_EXPR = 0.4611
           F1_AU@0.5 = 0.0000
          F1_AU_best = 0.0000
           t_AU_best = 0.5000
           P_MTL@0.5 = 0.8564
          P_MTL_best = 0.8564
             variant = f4_xattn
    trainable_params = 1720470.0000
[stage3] wrote C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\interim\crema_f5_lmf_on_ravdess\metrics.md
               ccc_V = 0.7028
               ccc_A = 0.3968
              CCC_VA = 0.5498
       F1_EXPR_macro = 0.3411
            ACC_EXPR = 0.4437
           F1_AU@0.5 = 0.0000
          F1_AU_best = 0.0000
           t_AU_best = 0.5000
           P_MTL@0.5 = 0.8909
          P_MTL_best = 0.8909
             variant = f5_lmf
    trainable_params = 597914.0000


c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [5]:
df_cross = pd.DataFrame([
    {"run": n, **{k: m.get(k, '') for k in COLS}}
    for n, m in cross_results.items()
])
(RESULTS / "ablation_cross_dataset.md").write_text(
    "# CREMA-D -> RAVDESS zero-shot (RH5)\n\n"
    "Checkpoint trained on CREMA-D train split; evaluated on RAVDESS val split "
    "with no adaptation.\n\n" +
    df_cross.to_markdown(index=False, floatfmt=".4f") + "\n",
    encoding="utf-8",
)
df_cross

,run,variant,ccc_V,ccc_A,CCC_VA,F1_EXPR_macro,F1_AU@0.5,P_MTL@0.5,trainable_params
0,crema_visual_only,visual_only,0.737235,0.398407,0.567821,0.390685,0.0,0.958506,179706
1,crema_audio_only,audio_only,0.056065,0.085891,0.070978,0.163129,0.0,0.234107,142998
2,crema_f1_concat,f1_concat,0.673480,0.273201,0.473340,0.349615,0.0,0.822956,995690
3,crema_f2_blend,f2_blend,0.723033,0.288729,0.505881,0.360708,0.0,0.866589,322708
4,crema_f3_gate,f3_gate,0.676059,0.380767,0.528413,0.364932,0.0,0.893345,624168
5,crema_f4_xattn,f4_xattn,0.695799,0.274279,0.485039,0.371395,0.0,0.856434,1720470
6,crema_f5_lmf,f5_lmf,0.702811,0.396813,0.549812,0.341092,0.0,0.890904,597914


## Ablation 3: RAVDESS-trained mirror (upper bound for RH5)

To know how much cross-dataset transfer costs, compare against models trained on RAVDESS directly. Just two anchors are enough for the table -- `visual_only` (to isolate the visual branch) and `f1_concat` (same fusion as the anchor bimodal).

In [6]:
rav_results = {}
for name in ["ravdess_visual_only", "ravdess_f1_concat"]:
    cfg = OmegaConf.load(CONFIGS / f"{name}.yaml")
    ck = train_or_skip(cfg)
    rav_results[name] = evaluate_variant(
        checkpoint=ck, config_path=CONFIGS / f"{name}.yaml",
    )
df_rav = pd.DataFrame([
    {"run": n, **{k: m.get(k, '') for k in COLS}}
    for n, m in rav_results.items()
])
(RESULTS / "ablation_ravdess_trained.md").write_text(
    "# RAVDESS-trained anchors (upper bound for RH5)\n\n" +
    df_rav.to_markdown(index=False, floatfmt=".4f") + "\n",
    encoding="utf-8",
)
df_rav

[fusion=visual_only] trainable params = 179,706
[visual_only] ep   1  train=1.9502  val=1.9597  best=1.9597  *
[visual_only] ep   2  train=1.3635  val=1.8458  best=1.8458  *
[visual_only] ep   3  train=1.1959  val=1.8113  best=1.8113  *
[visual_only] ep   4  train=1.0997  val=1.8159  best=1.8113
[visual_only] ep   5  train=1.0301  val=1.8135  best=1.8113
[visual_only] ep   6  train=0.9764  val=1.8199  best=1.8113
[visual_only] ep   7  train=0.9321  val=1.8443  best=1.8113
[visual_only] ep   8  train=0.8946  val=1.8501  best=1.8113
[visual_only] ep   9  train=0.8611  val=1.8611  best=1.8113
[visual_only] ep  10  train=0.8321  val=1.8785  best=1.8113
[visual_only] ep  11  train=0.8056  val=1.8968  best=1.8113
[visual_only] ep  12  train=0.7832  val=1.9048  best=1.8113
[visual_only] ep  13  train=0.7615  val=1.9193  best=1.8113
[visual_only] ep  14  train=0.7413  val=1.9286  best=1.8113
[visual_only] ep  15  train=0.7242  val=1.9425  best=1.8113
[fusion=visual_only] checkpoint -> results\

c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


[stage3] wrote results\interim\ravdess_visual_only\metrics.md
               ccc_V = 0.7654
               ccc_A = 0.5091
              CCC_VA = 0.6373
       F1_EXPR_macro = 0.4608
            ACC_EXPR = 0.5623
           F1_AU@0.5 = 0.0000
          F1_AU_best = 0.0000
           t_AU_best = 0.5000
           P_MTL@0.5 = 1.0981
          P_MTL_best = 1.0981
             variant = visual_only
    trainable_params = 179706.0000
[fusion=f1_concat] trainable params = 995,690
[f1_concat] ep   1  train=1.3847  val=1.8133  best=1.8133  *
[f1_concat] ep   2  train=0.7679  val=1.9880  best=1.8133
[f1_concat] ep   3  train=0.5438  val=2.0993  best=1.8133
[f1_concat] ep   4  train=0.4210  val=2.4480  best=1.8133
[f1_concat] ep   5  train=0.3461  val=2.4952  best=1.8133
[f1_concat] ep   6  train=0.2924  val=2.4975  best=1.8133
[f1_concat] ep   7  train=0.2504  val=2.7270  best=1.8133
[f1_concat] ep   8  train=0.2232  val=2.7604  best=1.8133
[f1_concat] ep   9  train=0.1950  val=2.7327  best=1.81

c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,run,variant,ccc_V,ccc_A,CCC_VA,F1_EXPR_macro,F1_AU@0.5,P_MTL@0.5,trainable_params
0,ravdess_visual_only,visual_only,0.765409,0.509119,0.637264,0.460797,0.0,1.098060,179706
1,ravdess_f1_concat,f1_concat,0.722060,0.559402,0.640731,0.481328,0.0,1.122059,995690


## Combined summary

Single Markdown block for direct paste into the report.

In [7]:
out = RESULTS / "ablations_summary.md"
with out.open("w", encoding="utf-8") as fh:
    fh.write("# Interim ablations -- combined summary\n\n")
    fh.write("## Visual backbone\n\n")
    fh.write(df_bb.to_markdown(index=False, floatfmt=".4f"))
    fh.write("\n\n## Cross-dataset (CREMA-D -> RAVDESS, zero-shot)\n\n")
    fh.write(df_cross.to_markdown(index=False, floatfmt=".4f"))
    fh.write("\n\n## RAVDESS-trained anchors\n\n")
    fh.write(df_rav.to_markdown(index=False, floatfmt=".4f"))
    fh.write("\n")
print("wrote", out)

wrote C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\interim\ablations_summary.md
